# 第04章 色彩空间类型转换（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 4 章跟随代码。**全量收录原书代码（例 4.1～例 4.13）**，缺失例题已按书补全，代码以网页笔记 docs/opencv/ch04.md 为准（含笔误修正与参数修正）。
> cv2.imshow / cv_show 需在本地 Python 环境运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 imread 返回 None。
> 素材替换：原书 lenacolor.png、barbara.bmp 用 picture1.jpg 顶替，lesson2.jpg 用 picture3.jpg 顶替；例 4.9 的原书 opencv.jpg（OpenCV logo）素材缺失，用 picture1.jpg 顶替，代码逻辑不变。

In [ ]:
import cv2

# cv_show 辅助函数：原书跟随代码沿用的显示封装（本地运行显示窗口用，弹窗显示后按任意键关闭）
def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


## 4.1 色彩空间基础

色彩空间（color space）也称颜色空间、颜色模型、彩色模型等，是表述颜色的模式。每个色彩空间都有自己擅长的处理领域，为了更方便地处理某个具体问题，就要做色彩空间类型转换——把图像从一个色彩空间转换到另外一个色彩空间。常见的色彩空间：

- **GRAY**：灰度图像，通常指 8 位灰度图，256 个灰度级。RGB→GRAY 按 `Gray = 0.299·R + 0.587·G + 0.114·B` 加权平均（人眼对绿色最敏感、蓝色最弱）；GRAY→RGB 时三个通道的值都相同；
- **XYZ**：CIE（国际照明委员会）定义的色彩空间，更便于计算，可与 RGB 相互转换；
- **YCrCb**：Y 代表光源的亮度，色度信息保存在 Cr（红色分量）和 Cb（蓝色分量）中——人眼对颜色的敏感度低于对亮度的敏感度；
- **HSV**：色调（Hue）、饱和度（Saturation）、亮度（Value）三要素，面向视觉感知的颜色模型（4.4 节详细讨论）；
- **HLS**：三要素为色调 H、光亮度/明度 L（Lightness）、饱和度 S，用"明度"顶替了 HSV 的"亮度"；
- **CIEL\*a\*b\* 与 CIEL\*u\*v\***：均匀色彩空间——人所观察到的两种颜色的区别程度，与两种颜色在空间中对应点的欧氏距离成正比；
- **Bayer**：广泛应用在 CCD 和 CMOS 相机中，从单平面 R、G、B 交错表中获取彩色图像。

> 本节为概念介绍，无例题；类型转换函数见 4.2 节，转换实例从 4.3 节开始。

## 4.2 类型转换函数

OpenCV 用 `cv2.cvtColor()` 实现色彩空间变换，它能在多个色彩空间之间转换：

```python
dst = cv2.cvtColor( src, code [, dstCn] )
```

- **src**：原始输入图像，可以是 8 位无符号、16 位无符号或单精度浮点数图像；
- **code**：色彩空间转换码，形如 `cv2.COLOR_BGR2GRAY`（BGR→灰度）、`cv2.COLOR_BGR2HSV`（BGR→HSV）、`cv2.COLOR_BGR2RGB`、`cv2.COLOR_HSV2BGR`、`cv2.COLOR_HSV2RGB`、`cv2.COLOR_RGB2BGR` 等，BGR2X 与 X2BGR 成对出现；
- **dstCn**：目标图像的通道数，默认 0 时由输入图像和 code 自动推得。

注意 BGR 与传统 RGB 的字节顺序不同：BGR 的第 1 个字节存蓝色（Blue）、第 2 个存绿色（Green）、第 3 个存红色（Red）。色彩空间转换的值域约定：8 位图 [0, 255]、16 位图 [0, 65535]、浮点图 [0.0, 1.0]。很多色彩空间的值不落在 [0, 255] 内，需要映射：HSV/HLS 的色调 H 要**除以 2** 压进 [0, 180)；CIEL\*a\*b\* 的 a、b 通道要**加 127** 平移到 [0, 255]。由于计算过程存在四舍五入，转换过程**并不是精准可逆的**。

> 本节无例题，转换实例见 4.3 节。

## 4.3 类型转换实例

### 4.3.1 通过数组观察转换效果

用小尺寸随机数组模拟图像，作为 `cv2.cvtColor()` 的处理对象，方便观察转换前后的数值变化。本节包含例 4.1～例 4.5，本小节为例 4.1～例 4.3。

### 例 4.1 · BGR 转 GRAY（数组模拟）

生成 2×4×3 的随机数组模拟 BGR 图像，转灰度并手工验算像素点 (1,0)：`Gray = 0.114·B + 0.587·G + 0.299·R`。加权求和得小数（如 120.632）后**四舍五入**取整，才是 `cvtColor` 输出的 8 位整数值 121——函数与公式一致。

In [ ]:
# ===== 4.3.1 例 4.1：BGR -> GRAY，并手工验算一个点 =====
import cv2
import numpy as np
img = np.random.randint(0, 256, size=[2, 4, 3], dtype=np.uint8)
rst = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("img=\n", img)
print("rst=\n", rst)
print("像素点(1,0)直接计算得到的值=",
      img[1, 0, 0] * 0.114 + img[1, 0, 1] * 0.587 + img[1, 0, 2] * 0.299)
print("像素点(1,0)使用公式cv2.cvtColor()转换值=", rst[1, 0])

### 例 4.2 · GRAY 转 BGR（数组模拟）

对 2×4 的随机灰度数组执行 `cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)`：输出三个通道的值完全相同（R = G = B = Gray），得到的是"灰度样"的 BGR 图——GRAY→BGR 复原不了颜色。

In [ ]:
# ===== 4.3.1 例 4.2：GRAY -> BGR，三个通道填同一个值 =====
import cv2
import numpy as np
img = np.random.randint(0, 256, size=[2, 4], dtype=np.uint8)
rst = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
print("img=\n", img)
print("rst=\n", rst)

### 例 4.3 · BGR 与 RGB 相互转换（数组模拟）

`COLOR_BGR2RGB` 把首尾两个通道互换（G 不动），再经 `COLOR_RGB2BGR` 转回，结果与原数组逐像素相同——BGR 与 RGB 互转只是 R、B 通道位置对调。

In [ ]:
# ===== 4.3.1 例 4.3：BGR <-> RGB，首尾通道互换 =====
import cv2
import numpy as np
img = np.random.randint(0, 256, size=[2, 4, 3], dtype=np.uint8)
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
print("img=\n", img)
print("rgb=\n", rgb)
print("bgr=\n", bgr)

**补充演示**（跟随代码原有）：在真实图像上做 GRAY→BGR 与 BGR→RGB——`picture2.png` 以灰度方式读入后转 BGR（三个通道同值，看起来仍是灰度图），`picture1.jpg` 转 RGB 后直接显示会红蓝互换、整体偏色。

In [202]:
#灰度转BGR
img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)
cv_show("gray",img)
rst=cv2.cvtColor(img,cv2.COLOR_GRAY2BGR)
cv_show("gray2bgr",rst)
#BGR转RGB
img=cv2.imread("picture1.jpg",cv2.IMREAD_UNCHANGED)
print(img)
cv_show("bgr",img)
rst=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
cv_show("bgr2rgb",rst)

[[[250 253 228]
  [250 253 228]
  [252 251 230]
  ...
  [230 233 217]
  [230 233 217]
  [230 233 217]]

 [[250 253 228]
  [250 253 228]
  [252 251 230]
  ...
  [230 233 217]
  [230 233 217]
  [230 233 217]]

 [[252 251 230]
  [252 251 230]
  [252 251 230]
  ...
  [230 233 217]
  [230 233 217]
  [230 233 217]]

 ...

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [160 158 147]
  [160 158 147]
  [160 158 147]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [160 158 147]
  [160 158 147]
  [160 158 147]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [160 158 147]
  [160 158 147]
  [160 158 147]]]


### 4.3.2 图像处理实例

### 例 4.4 · BGR 与灰度相互转换（真实图像）

对 `picture1.jpg`（原书 lenacolor.png）做 BGR→GRAY→BGR，打印各阶段 shape：三维 (1080, 1920, 3) → 二维 (1080, 1920) → 三维 (1080, 1920, 3)。注意 `COLOR_GRAY2BGR` 得到的"RGB 图像"三个通道值完全一样，看起来仍是灰度图。

In [ ]:
# ===== 4.3.2 例 4.4：shape 在转换前后的变化 =====
import cv2
lena = cv2.imread("picture1.jpg")                    # 原书 lenacolor.png
gray = cv2.cvtColor(lena, cv2.COLOR_BGR2GRAY)
rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
#==========打印shape============
print("lena.shape=", lena.shape)
print("gray.shape=", gray.shape)
print("rgb.shape=", rgb.shape)
#==========显示效果============
cv2.imshow("lena", lena)
cv2.imshow("gray", gray)
cv2.imshow("rgb", rgb)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 4.5 · BGR 转 RGB（真实图像）

`cv2.cvtColor(lena, cv2.COLOR_BGR2RGB)` 后用 imshow 直接显示：右图红蓝互换、整体偏蓝紫——因为 imshow 按 BGR 顺序解释通道。第 2 章"merge 通道顺序一换颜色就变"的现象，这里是第 3 种触发方式。

In [ ]:
# ===== 4.3.2 例 4.5：BGR -> RGB 后直接显示会偏色 =====
import cv2
lena = cv2.imread("picture1.jpg")                    # 原书 lenacolor.png
rgb = cv2.cvtColor(lena, cv2.COLOR_BGR2RGB)
cv2.imshow("lena", lena)
cv2.imshow("rgb", rgb)
cv2.waitKey()
cv2.destroyAllWindows()

## 4.4 HSV 色彩空间讨论

RGB 是被广泛接受的色彩空间，但过于抽象——不能直接通过数值感知具体色彩。HSV 从心理学和视觉的角度出发，用三要素描述颜色，更符合人的直观感知。

### 4.4.1 基础知识

- **色调 H**：光的颜色类别，取值 [0, 360]。8 位图每个像素只有 2^8 = 256 个灰度级，OpenCV 把角度值**除以 2** 映射到 [0, 180]：0 红、30 黄、60 绿、90 青、120 蓝、150 品红；
- **饱和度 S**：色彩的纯度，取值 [0, 1]（8 位图映射到 [0, 255]）。灰度颜色的 R、G、B 成分相等，饱和度为 0；饱和度很低时计算所得的色调不可靠；
- **亮度 V**：人眼感受到的明暗程度，取值 [0, 1]（8 位图映射到 [0, 255]）。亮度值越大图像越亮，亮度值为 0 时是纯黑色。

确定值范围后，就可以直接在图像的 H 通道内查找对应的值找到特定颜色——例如 H 通道内值为 120 的像素点对应蓝色；进一步分析出各种对象对应的 HSV 值，就能查找不同的对象。

### 4.4.2 获取指定颜色

获取 RGB 颜色在 HSV 内对应值的方式有多种（图像编辑软件、在线网站等），但软件或网站给出的 HSV 值还要**进一步映射**才与 OpenCV 内的值一致。更直接的办法是用程序测——把指定颜色做成单像素数组转一次 HSV，见例 4.6。

### 例 4.6 · 测试 RGB 中不同颜色转换到 HSV 后的对应值

用 `np.zeros([1, 1, 3], dtype=np.uint8)` 生成单像素图像，对通道赋值指定颜色（B 通道 255 = 蓝、G 通道 255 = 绿、R 通道 255 = 红），再转 HSV 打印。结果与表 4-4 一致：蓝 [120 255 255]、绿 [60 255 255]、红 [0 255 255]——这是 OpenCV 官方教程推荐的单点测色法。

In [ ]:
# ===== 4.4.2 例 4.6：单像素测三个原色的 HSV 值 =====
import cv2
import numpy as np
#=========测试一下OpenCV中蓝色的HSV模式值=============
imgBlue = np.zeros([1, 1, 3], dtype=np.uint8)
imgBlue[0, 0, 0] = 255
Blue = imgBlue
BlueHSV = cv2.cvtColor(Blue, cv2.COLOR_BGR2HSV)
print("Blue=\n", Blue)
print("BlueHSV=\n", BlueHSV)
#=========测试一下OpenCV中绿色的HSV模式值=============
imgGreen = np.zeros([1, 1, 3], dtype=np.uint8)
imgGreen[0, 0, 1] = 255
Green = imgGreen
GreenHSV = cv2.cvtColor(Green, cv2.COLOR_BGR2HSV)
print("Green=\n", Green)
print("GreenHSV=\n", GreenHSV)
#=========测试一下OpenCV中红色的HSV模式值=============
imgRed = np.zeros([1, 1, 3], dtype=np.uint8)
imgRed[0, 0, 2] = 255
Red = imgRed
RedHSV = cv2.cvtColor(Red, cv2.COLOR_BGR2HSV)
print("Red=\n", Red)
print("RedHSV=\n", RedHSV)

### 4.4.3 标记指定颜色

在 HSV 色彩空间中，颜色的差异主要体现在 H（色调 Hue）通道值的不同上——对 H 通道值进行筛选，就能筛选出特定的颜色。本小节包含例 4.7～例 4.9，分三步：

**1．通过 inRange 函数锁定特定值**

```python
dst = cv2.inRange( src, lowerb, upperb )
```

返回值 dst 与 src 等大小：对应位置的值落在区间 [lowerb, upperb] 内为 **255**，否则为 **0**——返回的 mask 是一个掩码数组。

**2．通过基于掩码的按位与显示 ROI**：`cv2.bitwise_and(img, img, mask=mask)` 只在掩码非零的位置执行运算，其余位置置 0——掩码只看非零，不要求 255。

**3．显示特定颜色值**：实际提取颜色时往往提取一个**颜色区间**而非特定值——如蓝色 H=120，通常把 120 附近半径 10 左右的区间作为提取范围；S、V 的取值范围一般是 [100, 255]（太低时算出的色调不可靠）。

> 原书笔误：本节开头写"H 通道（饱和度Hue 通道）"，H 是色调（Hue）不是饱和度（Saturation）。另注意红色横跨 H=0 两侧：[0, 10] 与 [170, 180) 两段各做一次 inRange，再用 `cv2.bitwise_or(mask1, mask2)` 合并。

### 例 4.7 · 将图像内 [100, 200] 范围内的值标注出来

对 5×5 随机数组执行 `cv2.inRange(img, 100, 200)`：区间内的值对应位置为 255、区间外为 0，逐位置对照 img 与 mask 即可验证。

In [ ]:
# ===== 4.4.3 例 4.7：inRange 的基本行为 =====
import cv2
import numpy as np
img = np.random.randint(0, 256, size=[5, 5], dtype=np.uint8)
min = 100
max = 200
mask = cv2.inRange(img, min, max)
print("img=\n", img)
print("mask=\n", mask)

**补充演示**（跟随代码原有）：把 `picture1.jpg` 转到 HSV 后用 `inRange` 取 (0, 100, 100)~(60, 255, 255) 区间——演示 inRange 作用于真实图像 HSV 的效果。

In [204]:
import cv2
import numpy as np

img=cv2.imread("picture1.jpg")
img1=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
print(img1)
cv_show("img1",img1)

dst=cv2.inRange(img1,(0,100,100),(60,255,255))
cv_show("dst",dst)

[[[ 86  25 253]
  [ 86  25 253]
  [ 91  22 252]
  ...
  [ 84  18 233]
  [ 84  18 233]
  [ 84  18 233]]

 [[ 86  25 253]
  [ 86  25 253]
  [ 91  22 252]
  ...
  [ 84  18 233]
  [ 84  18 233]
  [ 84  18 233]]

 [[ 91  22 252]
  [ 91  22 252]
  [ 91  22 252]
  ...
  [ 84  18 233]
  [ 84  18 233]
  [ 84  18 233]]

 ...

 [[  0   0 255]
  [  0   0 255]
  [  0   0 255]
  ...
  [ 95  21 160]
  [ 95  21 160]
  [ 95  21 160]]

 [[  0   0 255]
  [  0   0 255]
  [  0   0 255]
  ...
  [ 95  21 160]
  [ 95  21 160]
  [ 95  21 160]]

 [[  0   0 255]
  [  0   0 255]
  [  0   0 255]
  ...
  [ 95  21 160]
  [ 95  21 160]
  [ 95  21 160]]]


### 例 4.8 · 正常显示图像内的感兴趣区域（ROI），其余区域显示为黑色

全 9 数组 + 手工掩码（两块 ROI），`cv2.bitwise_and(img, img, mask=mask)` 后只有掩码非零的位置保留原值，其余清零。注意掩码值是 1 也能生效——**掩码只看非零，不要求 255**。

In [ ]:
# ===== 4.4.3 例 4.8：掩码控制按位与，只留 ROI =====
import cv2
import numpy as np
img = np.ones([5, 5], dtype=np.uint8) * 9
mask = np.zeros([5, 5], dtype=np.uint8)
mask[0:3, 0] = 1
mask[2:5, 2:4] = 1
roi = cv2.bitwise_and(img, img, mask=mask)
print("img=\n", img)
print("mask=\n", mask)
print("roi=\n", roi)

### 例 4.9 · 分别提取图像内的红色、绿色、蓝色

三段代码结构完全一样，只有阈值不同：先 `inRange` 找到指定颜色区域得掩码，再用掩码控制的按位与把目标抠出来。颜色区间按 [H-10, S下界, V下界] ~ [H+10, 255, 255] 设定：蓝 (110, 50, 50)~(130, 255, 255)、绿 (50, 50, 50)~(70, 255, 255)、红 (0, 50, 50)~(30, 255, 255)。

> 原书正文说 S、V 的范围"一般是 [100, 255]"，例 4.9 代码的下界却写成 50、红色区间上限也偏宽（30）——代码能跑，理解时以正文为准，实战时按图片效果调参。原书 opencv.jpg（OpenCV logo）素材缺失，本例用 picture1.jpg 顶替。

In [ ]:
# ===== 4.4.3 例 4.9：inRange + 掩码按位与，抠出蓝/绿/红 =====
import cv2
import numpy as np
opencv = cv2.imread("picture1.jpg")     # 原书 opencv.jpg（OpenCV logo）素材缺失，实跑用 picture1.jpg 顶替
hsv = cv2.cvtColor(opencv, cv2.COLOR_BGR2HSV)
cv2.imshow('opencv', opencv)
#=============指定蓝色值的范围=============
minBlue = np.array([110, 50, 50])
maxBlue = np.array([130, 255, 255])
#确定蓝色区域
mask = cv2.inRange(hsv, minBlue, maxBlue)
#通过掩码控制的按位与运算，锁定蓝色区域
blue = cv2.bitwise_and(opencv, opencv, mask=mask)
cv2.imshow('blue', blue)
#=============指定绿色值的范围=============
minGreen = np.array([50, 50, 50])
maxGreen = np.array([70, 255, 255])
#确定绿色区域
mask = cv2.inRange(hsv, minGreen, maxGreen)
#通过掩码控制的按位与运算，锁定绿色区域
green = cv2.bitwise_and(opencv, opencv, mask=mask)
cv2.imshow('green', green)
#=============指定红色值的范围=============
minRed = np.array([0, 50, 50])
maxRed = np.array([30, 255, 255])
#确定红色区域
mask = cv2.inRange(hsv, minRed, maxRed)
#通过掩码控制的按位与运算，锁定红色区域
red = cv2.bitwise_and(opencv, opencv, mask=mask)
cv2.imshow('red', red)
cv2.waitKey()
cv2.destroyAllWindows()

### 4.4.4 标记肤色

### 例 4.10 · 提取一幅图像内的肤色部分

把标注范围推广到特定对象：通过分析估算出肤色在 HSV 空间内的范围值（色调 [5, 170]、饱和度 [25, 166]），再在 HSV 空间内筛选出该范围内的值。与例 4.9 的区别在于**两个通道各做一个掩码再相与**（`hueMask & satMask`）——色调落在肤色区间、饱和度也落在肤色区间，两个条件同时满足才算命中。

In [ ]:
# ===== 4.4.4 例 4.10：H 掩码与 S 掩码相与，标记肤色 =====
import cv2
img = cv2.imread("picture3.jpg")          # 原书 lesson2.jpg（人像）
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)
minHue = 5
maxHue = 170
hueMask = cv2.inRange(h, minHue, maxHue)
minSat = 25
maxSat = 166
satMask = cv2.inRange(s, minSat, maxSat)
mask = hueMask & satMask
roi = cv2.bitwise_and(img, img, mask=mask)
cv2.imshow("img", img)
cv2.imshow("ROI", roi)
cv2.waitKey()
cv2.destroyAllWindows()

### 4.4.5 实现艺术效果

### 例 4.11 · 调整 HSV 色彩空间内 V 通道的值

在 HSV 色彩空间内调整分量值能生成有趣的效果，一些图像处理软件正是靠调 HSV 实现各种艺术效果的。例 4.11 把 H 通道和 S 通道保持不变，将 V 通道的值全部调成 255（最亮）：暗部整体提亮、高光区接近泛白。流程值得记：**拆通道 → 改某一通道 → merge 回去 → 转回 BGR**。改成其他值是别的味道——跟随代码原版把 V 置 120，得到的是压暗版。

In [ ]:
# ===== 4.4.5 例 4.11：V 通道全部拉满的艺术效果 =====
import cv2
img = cv2.imread("picture1.jpg")          # 原书 barbara.bmp
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)
v[:, :] = 255
newHSV = cv2.merge([h, s, v])
art = cv2.cvtColor(newHSV, cv2.COLOR_HSV2BGR)
cv2.imshow("img", img)
cv2.imshow("art", art)
cv2.waitKey()
cv2.destroyAllWindows()

## 4.5 alpha 通道

在 RGB 三个通道的基础上加一个 A 通道（**alpha 通道**，表示透明度），就得到四通道的 RGBA 色彩空间，PNG 图像是一种典型的 4 通道图像。alpha 通道的赋值范围是 [0, 1] 或 [0, 255]，表示从透明到不透明。本节包含例 4.12、例 4.13。

### 例 4.12 · 分析 alpha 通道的值

`cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)` 转出后 A 是 alpha 通道，**默认值为 255**；split 出 alpha 置 125，再 `cv2.merge([b, g, r, a])` 合并——等价于把 BGRA 图像的 alpha 值改成 125。

In [ ]:
# ===== 4.5 例 4.12：BGR -> BGRA，观察并修改 alpha =====
import cv2
import numpy as np
img = np.random.randint(0, 256, size=[2, 3, 3], dtype=np.uint8)
bgra = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
print("img=\n", img)
print("bgra=\n", bgra)
b, g, r, a = cv2.split(bgra)
print("a=\n", a)
a[:, :] = 125
bgra = cv2.merge([b, g, r, a])
print("bgra=\n", bgra)

### 例 4.13 · 对图像的 alpha 通道进行处理

alpha 分别为 255（bgra）、125、0 的三幅图，**imshow 显示时没有差别**（显示只认前三个通道），保存后才见分晓：PNG 能完整保留 4 通道，存成 JPEG 则直接丢掉 alpha（读回只剩 3 通道）；白底预览里 bgra125 像蒙了一层白雾，bgra0 其实是完全透明的（衬白底显白、衬黑底显黑）。

In [ ]:
# ===== 4.5 例 4.13：三种 alpha 值，显示无差别，保存见分晓 =====
import cv2
img = cv2.imread("picture3.jpg")          # 原书 lenacolor.png
bgra = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
b, g, r, a = cv2.split(bgra)
a[:, :] = 125
bgra125 = cv2.merge([b, g, r, a])
a[:, :] = 0
bgra0 = cv2.merge([b, g, r, a])
cv2.imshow("img", img)
cv2.imshow("bgra", bgra)
cv2.imshow("bgra125", bgra125)
cv2.imshow("bgra0", bgra0)
cv2.waitKey()
cv2.destroyAllWindows()
cv2.imwrite("bgra.png", bgra)
cv2.imwrite("bgra125.png", bgra125)
cv2.imwrite("bgra0.png", bgra0)

## 坑与陷阱（速查）

1. 8 位 HSV 的 H 范围是 [0, 180)，不是 [0, 360]——照"蓝色 240"去阈值永远抓不到，8 位图里蓝色在 120；
2. 两套色调值别混用：理论值 0/60/120/180/240/300 对应红黄绿青蓝品红，OpenCV 8 位存储值是 0/30/60/90/120/150；
3. `inRange` 必须作用在 HSV 图上，上下界按 (H, S, V) 顺序排；
4. S、V 下界别写 0——太低时算出的色调不可靠（正文建议 [100, 255]，例 4.9 代码用 50）；
5. 红色横跨 H=0 两侧，两段各 inRange 一次再 bitwise_or 合并；
6. 转换不是精准可逆（8 位量化 + 四舍五入），别用 `==` 验证往返；
7. alpha 显示无差别、存储才见真章：PNG 存得下 4 通道，JPEG 直接丢 alpha；
8. GRAY→BGR 复原不了颜色——三个通道填同一个灰度值，仍是"灰度样"图。